# Tutorial: Milestone 2 Verification (Rust Backend E2E)

Audience:
- Contributors validating Milestone 2 behavior on the Rust daemon.

Prerequisites:
- Run from repository root (`/Users/austin/GitHub/lucida`).
- `cargo`, `uv`, and notebook dependencies are installed.

Expected outcomes:
- Rust daemon starts and responds on `/healthz`.
- Session/view lifecycle routes work (`create/get/update`) with expected state transitions.
- Selector clamp/strict and patch errors match contract behavior.
- Rust milestone 2 parity test passes with frozen fixtures.


## Outline

1. Setup helpers and daemon controls
2. Start Rust daemon + build real OME-Zarr fixture
3. Exercise session/view/update flow and assert hash/version behavior
4. Run parity test and fixture coverage checks
5. Cleanup daemon and temporary artifacts


In [1]:
from __future__ import annotations

import json
import os
import shutil
import socket
import subprocess
import time
from pathlib import Path

import httpx
import numpy as np
import zarr

cwd = Path.cwd()
if (cwd / 'MIGRATION.md').exists():
    REPO_ROOT = cwd
elif (cwd.parent.parent / 'MIGRATION.md').exists():
    REPO_ROOT = cwd.parent.parent
else:
    raise AssertionError('Could not locate repository root containing MIGRATION.md')


def run(cmd: str, *, extra_env: dict[str, str] | None = None) -> str:
    env = dict(os.environ)
    if extra_env:
        env.update(extra_env)

    print(f"$ {cmd}")
    completed = subprocess.run(
        cmd,
        shell=True,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        capture_output=True,
    )

    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        if completed.stderr:
            print(completed.stderr)
        raise AssertionError(f"Command failed ({completed.returncode}): {cmd}")
    return completed.stdout


def find_free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(('127.0.0.1', 0))
        return int(sock.getsockname()[1])


def wait_for_healthz(base_url: str, process: subprocess.Popen[str], timeout_s: float = 45.0) -> None:
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise AssertionError('Rust daemon exited before becoming healthy.')
        try:
            resp = httpx.get(f"{base_url}/healthz", timeout=1.0)
            if resp.status_code == 200 and resp.json().get('status') == 'ok':
                return
        except Exception:
            pass
        time.sleep(0.2)
    raise AssertionError('Timed out waiting for Rust daemon /healthz.')


def create_sample_omezarr(path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    root = zarr.open_group(store=str(path), mode='w')

    shape0 = (1, 2, 4, 8, 10)
    shape1 = (1, 2, 2, 4, 5)
    data0 = np.arange(np.prod(shape0), dtype=np.uint16).reshape(shape0)
    data1 = np.arange(np.prod(shape1), dtype=np.uint16).reshape(shape1)

    root.create_array('0', data=data0, chunks=(1, 1, 2, 4, 5), overwrite=True)
    root.create_array('1', data=data1, chunks=(1, 1, 1, 2, 3), overwrite=True)

    root.attrs['multiscales'] = [
        {
            'name': 'primary',
            'axes': [
                {'name': 't', 'type': 't'},
                {'name': 'c', 'type': 'c'},
                {'name': 'z', 'type': 'z', 'unit': 'micron'},
                {'name': 'y', 'type': 'y', 'unit': 'micron'},
                {'name': 'x', 'type': 'x', 'unit': 'micron'},
            ],
            'datasets': [
                {
                    'path': '0',
                    'coordinateTransformations': [
                        {'type': 'scale', 'scale': [1, 1, 1, 1, 1]},
                        {'type': 'translation', 'translation': [0, 0, 0, 0, 0]},
                    ],
                },
                {
                    'path': '1',
                    'coordinateTransformations': [
                        {'type': 'scale', 'scale': [1, 1, 2, 2, 2]},
                    ],
                },
            ],
        }
    ]

    root.attrs['omero'] = {
        'channels': [
            {'index': 0, 'label': 'DNA', 'color': 'FF0000', 'window': {'start': 10, 'end': 400}},
            {'index': 1, 'label': 'RNA', 'color': '00FF00', 'window': {'start': 20, 'end': 200}},
        ]
    }

    return str(path)


print(f"Repository root: {REPO_ROOT}")


Repository root: /Users/austin/GitHub/lucida


## Step 1 - Start daemon and prepare dataset

Expectation:
- Rust daemon is healthy.
- Session and dataset open calls succeed.


In [2]:
port = find_free_port()
base_url = f"http://127.0.0.1:{port}"

daemon_env = dict(os.environ)
daemon_env['LUCIDA_DAEMON_ADDR'] = f"127.0.0.1:{port}"
daemon_process = subprocess.Popen(
    ['cargo', 'run', '-p', 'lucida-daemon', '--quiet'],
    cwd=REPO_ROOT,
    env=daemon_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    text=True,
)

wait_for_healthz(base_url, daemon_process)
print({'daemon_base_url': base_url})

notebook_tmp = REPO_ROOT / 'tmp' / 'notebook-milestone2'
if notebook_tmp.exists():
    shutil.rmtree(notebook_tmp)
notebook_tmp.mkdir(parents=True, exist_ok=True)

omezarr_uri = create_sample_omezarr(notebook_tmp / 'sample.zarr')

session_resp = httpx.post(
    f"{base_url}/session/create",
    json={'schema_version': 1},
    timeout=30.0,
)
assert session_resp.status_code == 200
session_payload = session_resp.json()
session_id = session_payload['session_id']
assert session_id.startswith('session_')

open_resp = httpx.post(
    f"{base_url}/dataset/open",
    json={'schema_version': 1, 'uri': omezarr_uri, 'session_id': session_id},
    timeout=30.0,
)
assert open_resp.status_code == 200
open_payload = open_resp.json()
dataset_id = open_payload['dataset_summary']['dataset_id']
assert dataset_id.startswith('ds_')

{'session_id': session_id, 'dataset_id': dataset_id, 'uri': open_payload['dataset_summary']['uri']}


{'daemon_base_url': 'http://127.0.0.1:50001'}


{'session_id': 'session_b3ae6fa538fe401f',
 'dataset_id': 'ds_bf4af3f9396a1280',
 'uri': 'file:///Users/austin/GitHub/lucida/tmp/notebook-milestone2/sample.zarr'}

## Step 2 - View lifecycle, selector normalization, and hash/version assertions

Expectation:
- `view/create` starts at `state_version = 0` with a non-empty hash.
- Successful updates increment version and change hash.
- Strict out-of-bounds and invalid patch errors do not mutate persisted view state.


In [3]:
view_create_resp = httpx.post(
    f"{base_url}/view/create",
    json={
        'schema_version': 1,
        'session_id': session_id,
        'dataset_id': dataset_id,
        'mode': '2d',
    },
    timeout=30.0,
)
assert view_create_resp.status_code == 200
view_create_payload = view_create_resp.json()
view_state = view_create_payload['view_state']
view_id = view_state['view_id']
initial_hash = view_state['state_hash']
assert view_state['state_version'] == 0
assert isinstance(initial_hash, str) and len(initial_hash) == 64

update_index_resp = httpx.post(
    f"{base_url}/view/update",
    json={
        'schema_version': 1,
        'session_id': session_id,
        'view_id': view_id,
        'patch': [
            {
                'op': 'replace',
                'path': '/selectors',
                'value': [{'axis': 'z', 'kind': 'index', 'index': 2, 'clamp': True}],
            }
        ],
    },
    timeout=30.0,
)
assert update_index_resp.status_code == 200
update_index_payload = update_index_resp.json()
assert update_index_payload['view_state']['state_version'] == 1
hash_after_index = update_index_payload['view_state']['state_hash']
assert hash_after_index != initial_hash

update_range_clamp_resp = httpx.post(
    f"{base_url}/view/update",
    json={
        'schema_version': 1,
        'session_id': session_id,
        'view_id': view_id,
        'patch': [
            {
                'op': 'replace',
                'path': '/selectors',
                'value': [{'axis': 'z', 'kind': 'range', 'start': 100, 'end_exclusive': 200, 'clamp': True}],
            }
        ],
    },
    timeout=30.0,
)
assert update_range_clamp_resp.status_code == 200
update_range_clamp_payload = update_range_clamp_resp.json()
assert update_range_clamp_payload['view_state']['state_version'] == 2
range_selector = next(item for item in update_range_clamp_payload['selectors_applied'] if item['axis'] == 'z')
assert range_selector['kind'] == 'range'
assert range_selector['start'] >= 0
assert range_selector['end_exclusive'] > range_selector['start']
hash_after_range = update_range_clamp_payload['view_state']['state_hash']
assert hash_after_range != hash_after_index

strict_oob_resp = httpx.post(
    f"{base_url}/view/update",
    json={
        'schema_version': 1,
        'session_id': session_id,
        'view_id': view_id,
        'patch': [
            {
                'op': 'replace',
                'path': '/selectors',
                'value': [{'axis': 'z', 'kind': 'index', 'index': 999, 'clamp': False}],
            }
        ],
    },
    timeout=30.0,
)
assert strict_oob_resp.status_code == 422
assert strict_oob_resp.json()['code'] == 'selector_out_of_bounds'

invalid_patch_resp = httpx.post(
    f"{base_url}/view/update",
    json={
        'schema_version': 1,
        'session_id': session_id,
        'view_id': view_id,
        'patch': [{'op': 'replace', 'path': '/selectors/999/index', 'value': 1}],
    },
    timeout=30.0,
)
assert invalid_patch_resp.status_code == 422
assert invalid_patch_resp.json()['code'] == 'invalid_patch'

view_get_resp = httpx.get(
    f"{base_url}/view/{view_id}",
    params={'session_id': session_id},
    timeout=30.0,
)
assert view_get_resp.status_code == 200
final_view_payload = view_get_resp.json()['view_state']
assert final_view_payload['state_version'] == 2
assert final_view_payload['state_hash'] == hash_after_range

{
    'view_id': view_id,
    'state_version': final_view_payload['state_version'],
    'state_hash': final_view_payload['state_hash'],
}


{'view_id': 'view_745fe71682794865',
 'state_version': 2,
 'state_hash': '1298fb8a59e012cdafc3ad2955bb82edf008f333c5b44877999b88c79633b912'}

## Step 3 - Rust parity test and fixture coverage

Expectation:
- Milestone 2 parity test passes against the Rust daemon harness.
- Frozen fixture includes required case names.


In [4]:
parity_out = run('uv run pytest tests/python/parity/test_milestone2_viewstate_parity.py -q')
assert '1 passed' in parity_out

fixture_path = REPO_ROOT / 'tests' / 'parity' / 'fixtures' / 'milestone2' / 'viewstate_corpus.json'
fixture_payload = json.loads(fixture_path.read_text(encoding='utf-8'))
case_names = [case['name'] for case in fixture_payload['cases']]

expected_names = {
    'session_create_success',
    'dataset_open_with_session_success',
    'view_create_success',
    'view_get_success',
    'view_update_success',
    'view_update_index_clamped_success',
    'view_update_range_clamped_success',
    'view_update_set_clamped_success',
    'view_update_selector_index_out_of_bounds_error',
    'view_update_selector_range_out_of_bounds_error',
    'view_update_selector_set_out_of_bounds_error',
    'view_update_invalid_patch_error',
    'view_create_unknown_dataset_error',
    'view_create_unsupported_mode_error',
    'view_get_wrong_session_error',
    'view_get_unknown_session_error',
}
missing = sorted(expected_names - set(case_names))
assert not missing, f'Missing fixture cases: {missing}'

{'case_count': len(case_names), 'first_five_cases': case_names[:5]}


$ uv run pytest tests/python/parity/test_milestone2_viewstate_parity.py -q


.                                                                        [100%]
1 passed in 0.90s



{'case_count': 17,
 'first_five_cases': ['session_create_success',
  'dataset_open_with_session_success',
  'view_create_success',
  'view_get_success',
  'view_update_success']}

## Step 4 - Cleanup

This cell stops the daemon and removes temporary fixture data.


In [5]:
if 'daemon_process' in globals() and daemon_process.poll() is None:
    daemon_process.terminate()
    try:
        daemon_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        daemon_process.kill()
        daemon_process.wait(timeout=10)

if 'notebook_tmp' in globals() and notebook_tmp.exists():
    shutil.rmtree(notebook_tmp)

{'cleanup': 'ok'}


{'cleanup': 'ok'}